# 10 Strength of Schedule Features

This notebook adds strength of schedule features to the NFL forecasting model.

The current best model uses scoring, recent-form, and Elo features. This notebook adds opponent strength features to help the model understand whether a team's previous performance came against strong or weak opponents.

Current best accuracy: 63.51%

In [1]:
# Import packages
import os
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import sys
sys.path.append("../src")

from data_loader import load_game_results
from feature_engineering import create_modeling_dataset
from elo import create_elo_features

In [2]:
# Load game results 
game_results = load_game_results("../data/processed/game_results_2018_2025.csv")

game_results.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0


In [3]:
# Create base modeling dataset
modeling_data = create_modeling_dataset(game_results)

modeling_data.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,away_last3_avg_point_diff,away_last3_win_pct,avg_points_scored_diff,avg_points_allowed_diff,avg_point_diff_diff,win_pct_diff,last3_avg_points_scored_diff,last3_avg_points_allowed_diff,last3_avg_point_diff_diff,last3_win_pct_diff
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
# Create Elo features
elo_features = create_elo_features(game_results)

elo_features.head()

,game_id,season,week,gameday,home_team,away_team,home_elo_before,away_elo_before,elo_diff,home_elo_with_hfa_diff,elo_home_win_prob,home_elo_after,away_elo_after
0,2018_01_ATL_PHI,2018,1,2018-09-06,PHI,ATL,1500.0,1500.0,0.0,55.0,0.578497,1508.430065,1491.569935
1,2018_01_BUF_BAL,2018,1,2018-09-09,BAL,BUF,1500.0,1500.0,0.0,55.0,0.578497,1508.430065,1491.569935
2,2018_01_PIT_CLE,2018,1,2018-09-09,CLE,PIT,1500.0,1500.0,0.0,55.0,0.578497,1498.430065,1501.569935
3,2018_01_CIN_IND,2018,1,2018-09-09,IND,CIN,1500.0,1500.0,0.0,55.0,0.578497,1488.430065,1511.569935
4,2018_01_TEN_MIA,2018,1,2018-09-09,MIA,TEN,1500.0,1500.0,0.0,55.0,0.578497,1508.430065,1491.569935


In [6]:
# Merge Elo features
elo_cols = [
    "game_id",
    "home_elo_before",
    "away_elo_before",
    "elo_diff",
    "home_elo_with_hfa_diff",
    "elo_home_win_prob"
]

modeling_data = modeling_data.merge(
    elo_features[elo_cols],
    on="game_id",
    how="left"
)

modeling_data.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,win_pct_diff,last3_avg_points_scored_diff,last3_avg_points_allowed_diff,last3_avg_point_diff_diff,last3_win_pct_diff,home_elo_before,away_elo_before,elo_diff,home_elo_with_hfa_diff,elo_home_win_prob
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497


In [7]:
# Create team-game rows for opponent strength
games = game_results.copy()

games["gameday"] = pd.to_datetime(games["gameday"])

games = games.sort_values(["season", "week", "gameday"]).reset_index(drop=True)

home_rows = games[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
].copy()

home_rows = home_rows.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_scored",
        "away_score": "points_allowed"
    }
)

home_rows["is_home"] = 1

away_rows = games[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "away_team",
        "home_team",
        "away_score",
        "home_score"
    ]
].copy()

away_rows = away_rows.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_scored",
        "home_score": "points_allowed"
    }
)

away_rows["is_home"] = 0

team_games = pd.concat([home_rows, away_rows], ignore_index=True)

team_games = team_games.sort_values(
    ["season", "week", "gameday", "team"]
).reset_index(drop=True)

team_games.head()

,season,week,game_id,gameday,team,opponent,points_scored,points_allowed,is_home
0,2018,1,2018_01_ATL_PHI,2018-09-06,ATL,PHI,12.0,18.0,0
1,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1
2,2018,1,2018_01_WAS_ARI,2018-09-09,ARI,WAS,6.0,24.0,1
3,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1
4,2018,1,2018_01_BUF_BAL,2018-09-09,BUF,BAL,3.0,47.0,0


In [8]:
# Add team result
team_games["team_won"] = (
    team_games["points_scored"] > team_games["points_allowed"]
).astype(int)

team_games.head()

,season,week,game_id,gameday,team,opponent,points_scored,points_allowed,is_home,team_won
0,2018,1,2018_01_ATL_PHI,2018-09-06,ATL,PHI,12.0,18.0,0,0
1,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,1
2,2018,1,2018_01_WAS_ARI,2018-09-09,ARI,WAS,6.0,24.0,1,0
3,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,1
4,2018,1,2018_01_BUF_BAL,2018-09-09,BUF,BAL,3.0,47.0,0,0


In [9]:
# Create opponent win percentage before each game
team_games = team_games.sort_values(
    ["team", "season", "week", "gameday"]
).reset_index(drop=True)

team_games["games_played_before"] = (
    team_games.groupby(["team", "season"]).cumcount()
)

team_games["wins_before"] = (
    team_games.groupby(["team", "season"])["team_won"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["wins_before"] = team_games["wins_before"].fillna(0)

team_games["team_win_pct_before"] = 0.0

mask = team_games["games_played_before"] > 0

team_games.loc[mask, "team_win_pct_before"] = (
    team_games.loc[mask, "wins_before"] /
    team_games.loc[mask, "games_played_before"]
)

team_games[
    [
        "season",
        "week",
        "team",
        "opponent",
        "games_played_before",
        "wins_before",
        "team_win_pct_before"
    ]
].head(20)

,season,week,team,opponent,games_played_before,wins_before,team_win_pct_before
0,2018,1,ARI,WAS,0,0.0,0.000000
1,2018,2,ARI,LA,1,0.0,0.000000
2,2018,3,ARI,CHI,2,0.0,0.000000
3,2018,4,ARI,SEA,3,0.0,0.000000
4,2018,5,ARI,SF,4,0.0,0.000000
5,2018,6,ARI,MIN,5,1.0,0.200000
6,2018,7,ARI,DEN,6,1.0,0.166667
7,2018,8,ARI,SF,7,1.0,0.142857
8,2018,10,ARI,KC,8,2.0,0.250000
9,2018,11,ARI,OAK,9,2.0,0.222222


In [10]:
# Create lookup table for opponent strength
team_strength_lookup = team_games[
    [
        "game_id",
        "team",
        "team_win_pct_before"
    ]
].copy()

team_strength_lookup.head()

,game_id,team,team_win_pct_before
0,2018_01_WAS_ARI,ARI,0.0
1,2018_02_ARI_LA,ARI,0.0
2,2018_03_CHI_ARI,ARI,0.0
3,2018_04_SEA_ARI,ARI,0.0
4,2018_05_ARI_SF,ARI,0.0


In [11]:
# Add opponent win percentage to each team-game row
team_games = team_games.merge(
    team_strength_lookup,
    left_on=["game_id", "opponent"],
    right_on=["game_id", "team"],
    how="left",
    suffixes=("", "_opponent_lookup")
)

team_games = team_games.rename(
    columns={
        "team_win_pct_before_opponent_lookup": "opponent_win_pct_before"
    }
)

team_games = team_games.drop(columns=["team_opponent_lookup"])

team_games[
    [
        "season",
        "week",
        "team",
        "opponent",
        "team_win_pct_before",
        "opponent_win_pct_before"
    ]
].head(20)

,season,week,team,opponent,team_win_pct_before,opponent_win_pct_before
0,2018,1,ARI,WAS,0.000000,0.000000
1,2018,2,ARI,LA,0.000000,1.000000
2,2018,3,ARI,CHI,0.000000,0.500000
3,2018,4,ARI,SEA,0.000000,0.333333
4,2018,5,ARI,SF,0.000000,0.250000
5,2018,6,ARI,MIN,0.200000,0.400000
6,2018,7,ARI,DEN,0.166667,0.333333
7,2018,8,ARI,SF,0.142857,0.142857
8,2018,10,ARI,KC,0.250000,0.888889
9,2018,11,ARI,OAK,0.222222,0.111111


In [12]:
# Calculate average opponent win percentage before each game
team_games = team_games.sort_values(
    ["team", "season", "week", "gameday"]
).reset_index(drop=True)

team_games["opponent_win_pct_sum_before"] = (
    team_games.groupby(["team", "season"])["opponent_win_pct_before"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["opponents_played_before"] = (
    team_games.groupby(["team", "season"]).cumcount()
)

team_games["opponent_win_pct_sum_before"] = (
    team_games["opponent_win_pct_sum_before"].fillna(0)
)

team_games["strength_of_schedule_before"] = 0.0

mask = team_games["opponents_played_before"] > 0

team_games.loc[mask, "strength_of_schedule_before"] = (
    team_games.loc[mask, "opponent_win_pct_sum_before"] /
    team_games.loc[mask, "opponents_played_before"]
)

team_games[
    [
        "season",
        "week",
        "team",
        "opponent",
        "opponents_played_before",
        "strength_of_schedule_before"
    ]
].head(20)

,season,week,team,opponent,opponents_played_before,strength_of_schedule_before
0,2018,1,ARI,WAS,0,0.000000
1,2018,2,ARI,LA,1,0.000000
2,2018,3,ARI,CHI,2,0.500000
3,2018,4,ARI,SEA,3,0.500000
4,2018,5,ARI,SF,4,0.458333
5,2018,6,ARI,MIN,5,0.416667
6,2018,7,ARI,DEN,6,0.413889
7,2018,8,ARI,SF,7,0.402381
8,2018,10,ARI,KC,8,0.369940
9,2018,11,ARI,OAK,9,0.427601


In [13]:
# Check one team
team_games[team_games["team"] == "KC"][
    [
        "season",
        "week",
        "team",
        "opponent",
        "team_win_pct_before",
        "opponent_win_pct_before",
        "strength_of_schedule_before"
    ]
].head(20)

,season,week,team,opponent,team_win_pct_before,opponent_win_pct_before,strength_of_schedule_before
2070,2018,1,KC,LAC,0.000000,0.000000,0.000000
2071,2018,2,KC,PIT,1.000000,0.000000,0.000000
2072,2018,3,KC,SF,1.000000,0.500000,0.000000
2073,2018,4,KC,DEN,1.000000,0.666667,0.166667
2074,2018,5,KC,JAX,1.000000,0.750000,0.291667
2075,2018,6,KC,NE,1.000000,0.600000,0.383333
2076,2018,7,KC,CIN,0.833333,0.666667,0.419444
2077,2018,8,KC,DEN,0.857143,0.428571,0.454762
2078,2018,9,KC,CLE,0.875000,0.250000,0.451488
2079,2018,10,KC,ARI,0.888889,0.250000,0.429101


In [14]:
# Create home strength of schedule features
home_sos = team_games[team_games["is_home"] == 1][
    [
        "game_id",
        "team",
        "strength_of_schedule_before",
        "opponent_win_pct_before"
    ]
].copy()

home_sos = home_sos.rename(
    columns={
        "team": "home_team_check",
        "strength_of_schedule_before": "home_strength_of_schedule_before",
        "opponent_win_pct_before": "home_current_opponent_win_pct_before"
    }
)

home_sos.head()

,game_id,home_team_check,home_strength_of_schedule_before,home_current_opponent_win_pct_before
0,2018_01_WAS_ARI,ARI,0.000000,0.000000
2,2018_03_CHI_ARI,ARI,0.500000,0.500000
3,2018_04_SEA_ARI,ARI,0.500000,0.333333
6,2018_07_DEN_ARI,ARI,0.413889,0.333333
7,2018_08_SF_ARI,ARI,0.402381,0.142857


In [15]:
# Create away strength of schedule features
away_sos = team_games[team_games["is_home"] == 0][
    [
        "game_id",
        "team",
        "strength_of_schedule_before",
        "opponent_win_pct_before"
    ]
].copy()

away_sos = away_sos.rename(
    columns={
        "team": "away_team_check",
        "strength_of_schedule_before": "away_strength_of_schedule_before",
        "opponent_win_pct_before": "away_current_opponent_win_pct_before"
    }
)

away_sos.head()

,game_id,away_team_check,away_strength_of_schedule_before,away_current_opponent_win_pct_before
1,2018_02_ARI_LA,ARI,0.000000,1.000000
4,2018_05_ARI_SF,ARI,0.458333,0.250000
5,2018_06_ARI_MIN,ARI,0.416667,0.400000
8,2018_10_ARI_KC,ARI,0.369940,0.888889
10,2018_12_ARI_LAC,ARI,0.395952,0.700000


In [16]:
# Merge strength of schedule features
modeling_data_sos = modeling_data.merge(
    home_sos,
    on="game_id",
    how="left"
)

modeling_data_sos = modeling_data_sos.merge(
    away_sos,
    on="game_id",
    how="left"
)

modeling_data_sos.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,away_elo_before,elo_diff,home_elo_with_hfa_diff,elo_home_win_prob,home_team_check,home_strength_of_schedule_before,home_current_opponent_win_pct_before,away_team_check,away_strength_of_schedule_before,away_current_opponent_win_pct_before
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,1500.0,0.0,55.0,0.578497,PHI,0.0,0.0,ATL,0.0,0.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,1500.0,0.0,55.0,0.578497,BAL,0.0,0.0,BUF,0.0,0.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,1500.0,0.0,55.0,0.578497,CLE,0.0,0.0,PIT,0.0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,1500.0,0.0,55.0,0.578497,IND,0.0,0.0,CIN,0.0,0.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,1500.0,0.0,55.0,0.578497,MIA,0.0,0.0,TEN,0.0,0.0


In [17]:
# Check team matches
modeling_data_sos[
    [
        "home_team",
        "home_team_check",
        "away_team",
        "away_team_check"
    ]
].head(10)

,home_team,home_team_check,away_team,away_team_check
0,PHI,PHI,ATL,ATL
1,BAL,BAL,BUF,BUF
2,CLE,CLE,PIT,PIT
3,IND,IND,CIN,CIN
4,MIA,MIA,TEN,TEN
5,MIN,MIN,SF,SF
6,NE,NE,HOU,HOU
7,NO,NO,TB,TB
8,NYG,NYG,JAX,JAX
9,LAC,LAC,KC,KC


In [18]:
# Drop check columns
modeling_data_sos = modeling_data_sos.drop(
    columns=["home_team_check", "away_team_check"]
)

In [19]:
# Create strength of schedule difference feature
modeling_data_sos["strength_of_schedule_diff"] = (
    modeling_data_sos["home_strength_of_schedule_before"] -
    modeling_data_sos["away_strength_of_schedule_before"]
)

modeling_data_sos["current_opponent_win_pct_diff"] = (
    modeling_data_sos["home_current_opponent_win_pct_before"] -
    modeling_data_sos["away_current_opponent_win_pct_before"]
)

modeling_data_sos[
    [
        "season",
        "week",
        "home_team",
        "away_team",
        "home_strength_of_schedule_before",
        "away_strength_of_schedule_before",
        "strength_of_schedule_diff",
        "current_opponent_win_pct_diff"
    ]
].head(20)

,season,week,home_team,away_team,home_strength_of_schedule_before,away_strength_of_schedule_before,strength_of_schedule_diff,current_opponent_win_pct_diff
0,2018,1,PHI,ATL,0.0,0.0,0.0,0.0
1,2018,1,BAL,BUF,0.0,0.0,0.0,0.0
2,2018,1,CLE,PIT,0.0,0.0,0.0,0.0
3,2018,1,IND,CIN,0.0,0.0,0.0,0.0
4,2018,1,MIA,TEN,0.0,0.0,0.0,0.0
5,2018,1,MIN,SF,0.0,0.0,0.0,0.0
6,2018,1,NE,HOU,0.0,0.0,0.0,0.0
7,2018,1,NO,TB,0.0,0.0,0.0,0.0
8,2018,1,NYG,JAX,0.0,0.0,0.0,0.0
9,2018,1,LAC,KC,0.0,0.0,0.0,0.0


In [20]:
# Check missing values
sos_cols = [
    "home_strength_of_schedule_before",
    "away_strength_of_schedule_before",
    "strength_of_schedule_diff",
    "home_current_opponent_win_pct_before",
    "away_current_opponent_win_pct_before",
    "current_opponent_win_pct_diff"
]

modeling_data_sos[sos_cols].isna().sum()

home_strength_of_schedule_before        0
away_strength_of_schedule_before        0
strength_of_schedule_diff               0
home_current_opponent_win_pct_before    0
away_current_opponent_win_pct_before    0
current_opponent_win_pct_diff           0
dtype: int64

In [21]:
# Save strength of schedule dataset
modeling_data_sos.to_csv(
    "../data/processed/modeling_dataset_sos_2018_2025.csv",
    index=False
)

print("Saved strength of schedule modeling dataset.")
print(modeling_data_sos.shape)

Saved strength of schedule modeling dataset.
(2227, 47)


In [22]:
# Choose features
features = [
    "avg_points_scored_diff",
    "avg_points_allowed_diff",
    "avg_point_diff_diff",
    "win_pct_diff",
    "last3_avg_points_scored_diff",
    "last3_avg_points_allowed_diff",
    "last3_avg_point_diff_diff",
    "last3_win_pct_diff",
    "elo_diff",
    "home_elo_with_hfa_diff",
    "elo_home_win_prob",
    "strength_of_schedule_diff",
    "current_opponent_win_pct_diff"
]

target = "home_team_won"

In [23]:
# Train on 2018–2024 and test on 2025
train_data = modeling_data_sos[
    modeling_data_sos["season"].between(2018, 2024)
].copy()

test_data = modeling_data_sos[
    modeling_data_sos["season"] == 2025
].copy()

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 1942
Testing rows: 285


In [24]:
# Train model
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [25]:
# Make predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [26]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Strength of schedule model accuracy: {accuracy:.2%}")
print("Previous best accuracy: 63.51%")

Strength of schedule model accuracy: 63.86%
Previous best accuracy: 63.51%


In [27]:
# Home-team baseline
home_team_baseline = [1] * len(y_test)

baseline_accuracy = accuracy_score(y_test, home_team_baseline)

print(f"Home-team baseline accuracy: {baseline_accuracy:.2%}")

Home-team baseline accuracy: 53.33%


In [28]:
# Classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.63      0.56      0.59       133
           1       0.65      0.71      0.68       152

    accuracy                           0.64       285
   macro avg       0.64      0.63      0.63       285
weighted avg       0.64      0.64      0.64       285



In [29]:
# Model coefficients
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
})

coefficients.sort_values("coefficient", ascending=False)

,feature,coefficient
10,elo_home_win_prob,0.449170
7,last3_win_pct_diff,0.445213
12,current_opponent_win_pct_diff,0.232870
0,avg_points_scored_diff,0.025744
2,avg_point_diff_diff,0.019742
1,avg_points_allowed_diff,0.006002
8,elo_diff,0.004973
5,last3_avg_points_allowed_diff,0.004091
4,last3_avg_points_scored_diff,0.001427
9,home_elo_with_hfa_diff,-0.001302


In [30]:
# Create results table
results = test_data[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_team_won",
        "home_strength_of_schedule_before",
        "away_strength_of_schedule_before",
        "strength_of_schedule_diff",
        "current_opponent_win_pct_diff"
    ]
].copy()

results["predicted_home_win"] = y_pred
results["home_win_probability"] = y_prob

results["predicted_winner"] = results.apply(
    lambda row: row["home_team"] if row["predicted_home_win"] == 1 else row["away_team"],
    axis=1
)

results["actual_winner"] = results.apply(
    lambda row: row["home_team"] if row["home_team_won"] == 1 else row["away_team"],
    axis=1
)

results["correct_prediction"] = (
    results["predicted_winner"] == results["actual_winner"]
)

results.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_strength_of_schedule_before,away_strength_of_schedule_before,strength_of_schedule_diff,current_opponent_win_pct_diff,predicted_home_win,home_win_probability,predicted_winner,actual_winner,correct_prediction
1942,2025,1,2025_01_DAL_PHI,2025-09-04,PHI,DAL,24.0,20.0,1,0.0,0.0,0.0,0.0,1,0.681056,PHI,PHI,True
1943,2025,1,2025_01_KC_LAC,2025-09-05,LAC,KC,27.0,21.0,1,0.0,0.0,0.0,0.0,0,0.301609,KC,LAC,False
1944,2025,1,2025_01_TB_ATL,2025-09-07,ATL,TB,20.0,23.0,0,0.0,0.0,0.0,0.0,0,0.438481,TB,TB,True
1945,2025,1,2025_01_CIN_CLE,2025-09-07,CLE,CIN,16.0,17.0,0,0.0,0.0,0.0,0.0,0,0.407171,CIN,CIN,True
1946,2025,1,2025_01_MIA_IND,2025-09-07,IND,MIA,33.0,8.0,1,0.0,0.0,0.0,0.0,0,0.491980,MIA,IND,False


In [31]:
# Accuracy by week
accuracy_by_week = (
    results.groupby("week")["correct_prediction"]
    .mean()
    .reset_index()
)

accuracy_by_week["accuracy_percent"] = (
    accuracy_by_week["correct_prediction"] * 100
).round(2)

accuracy_by_week = accuracy_by_week.drop(columns=["correct_prediction"])

accuracy_by_week

,week,accuracy_percent
0,1,68.75
1,2,75.00
2,3,75.00
3,4,56.25
4,5,35.71
5,6,60.00
6,7,80.00
7,8,69.23
8,9,57.14
9,10,78.57


In [32]:
# Save predictions
results.to_csv(
    "../data/predictions/sos_model_test_predictions.csv",
    index=False
)

print("Saved strength of schedule model predictions.")
print(results.shape)

Saved strength of schedule model predictions.
(285, 18)


In [33]:
# Final summary
print("Strength of schedule model accuracy:", f"{accuracy:.2%}")
print("Previous best accuracy: 63.51%")
print("Home-team baseline accuracy:", f"{baseline_accuracy:.2%}")
print("Number of test games:", len(results))
print("Correct predictions:", results["correct_prediction"].sum())
print("Incorrect predictions:", len(results) - results["correct_prediction"].sum())

Strength of schedule model accuracy: 63.86%
Previous best accuracy: 63.51%
Home-team baseline accuracy: 53.33%
Number of test games: 285
Correct predictions: 182
Incorrect predictions: 103


## Summary

In this notebook, I added strength of schedule features to the model.

These features estimate how strong a team's previous opponents were before each game. The idea is that team performance should be interpreted differently depending on the strength of the teams they have already played.

The model trained on 2018–2024 and tested on 2025, using the same realistic evaluation setup as the previous notebooks.

The next step is to compare the strength of schedule model accuracy to the previous best accuracy of 63.51%.